In [1]:
import os
import pathlib
import sys
import socket



hostname = socket.gethostname()
print(hostname)


# abs_path = str(pathlib.Path(hostname).parent.absolute())
abs_path = str(pathlib.Path(hostname).parent.absolute())

WORKDIR = abs_path[:- (len(abs_path.split('/')[-2] + abs_path.split('/')[-1])+1)]
DATA_DIR = "/mn/vann/franzihe/"


INPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, 'output')
UTILS_DIR = os.path.join(WORKDIR, 'utils')

sys.path.append(UTILS_DIR)
import functions as fct

ivan.uio.no


In [2]:
from imports import glob, xr, pd

In [3]:
year = 2023

In [4]:
hatpro = xr.open_mfdataset(sorted(glob(os.path.join(INPUT_DATA_DIR, f'hatpro/hatpro_new_20240716/level1/{year}/*/*/MWR*.nc'))))

In [5]:
hatpro["air_temperature"]

<xarray.DataArray 'air_temperature' (time: 1546319)> Size: 6MB
dask.array<concatenate, shape=(1546319,), dtype=float32, chunksize=(70271,), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 12MB 2023-02-17 ... 2023-03-16T08:59:59
Attributes:
    units:          K
    long_name:      Air temperature
    standard_name:  air_temperature

In [6]:
date = "2023-03-08"

In [7]:
temp = hatpro[['air_temperature']].sel(time="2023-03-08")

In [10]:
temp.time.values

array(['2023-03-08T00:00:00.000000000', '2023-03-08T00:01:58.000000000',
       '2023-03-08T00:01:59.000000000', ...,
       '2023-03-08T19:59:57.000000000', '2023-03-08T19:59:58.000000000',
       '2023-03-08T19:59:59.000000000'],
      shape=(35069,), dtype='datetime64[ns]')

In [29]:
temp["air_temperature"].attrs

{'units': 'K',
 'long_name': 'Air temperature',
 'standard_name': 'air_temperature'}

In [30]:
for date in pd.date_range(start="2023-02-17",end="2023-03-16"):
    temp = hatpro[['air_temperature']].sel(time=date.strftime("%Y-%m-%d"))
    
    units = temp["air_temperature"].attrs.get('units', '').lower()
    # Check and convert if Kelvin
    if 'kelvin' in units or 'k' in units:
        print(f"⚠️ Air temperature is in Kelvin. Converting to Celsius... {date}")
        temp['air_temperature'] = temp['air_temperature'] - 273.15
        temp['air_temperature'].attrs['units'] = 'Celsius'
        temp['air_temperature'].attrs['note'] = 'Converted from Kelvin to Celsius'
    elif 'celsius' in units or 'degc' in units or 'degree_celsius' in units or 'degree_c' in units:
        print(f"✅ Air temperature is already in Celsius. {date}")
    else:
        print(f"❓ Unrecognized units: {units} (no conversion applied). {date}")



    file_name = os.path.join(INPUT_DATA_DIR, f'hatpro/hatpro_new_20240716/level1/{year}/air_temperature_{date.strftime("%Y%m%d")}.nc')
    temp.to_netcdf(file_name)
    print(f"saved file to {file_name}")

⚠️ Air temperature is in Kelvin. Converting to Celsius... 2023-02-17 00:00:00
saved file to /mn/vann/franzihe/input/hatpro/hatpro_new_20240716/level1/2023/air_temperature_20230217.nc
⚠️ Air temperature is in Kelvin. Converting to Celsius... 2023-02-18 00:00:00
saved file to /mn/vann/franzihe/input/hatpro/hatpro_new_20240716/level1/2023/air_temperature_20230218.nc
⚠️ Air temperature is in Kelvin. Converting to Celsius... 2023-02-19 00:00:00
saved file to /mn/vann/franzihe/input/hatpro/hatpro_new_20240716/level1/2023/air_temperature_20230219.nc
⚠️ Air temperature is in Kelvin. Converting to Celsius... 2023-02-20 00:00:00
saved file to /mn/vann/franzihe/input/hatpro/hatpro_new_20240716/level1/2023/air_temperature_20230220.nc
⚠️ Air temperature is in Kelvin. Converting to Celsius... 2023-02-21 00:00:00
saved file to /mn/vann/franzihe/input/hatpro/hatpro_new_20240716/level1/2023/air_temperature_20230221.nc
⚠️ Air temperature is in Kelvin. Converting to Celsius... 2023-02-22 00:00:00
saved f

In [11]:
new_file = xr.open_dataset("/mn/vann/franzihe/input/hatpro/hatpro_new_20240716/level1/2023/air_temperature_20230219.nc")

In [12]:
new_file

<xarray.Dataset> Size: 802kB
Dimensions:          (time: 66837)
Coordinates:
  * time             (time) datetime64[ns] 535kB 2023-02-19T00:01:51 ... 2023...
Data variables:
    air_temperature  (time) float32 267kB ...
Attributes: (12/46)
    mwrpy_version:                                0.8.2
    processed:                                    28 Jun 2024 08:54:29 UTC
    conventions:                                  CF-1.8
    title:                                        HATPRO G4 MWR at Andoya Spa...
    history:                                      
    institution:                                  University of Oslo
    ...                                           ...
    relative_humidity_accuracy:                   3-5%
    air_pressure_accuracy:                        0.5 hPa
    rainfall_rate_accuracy:                       < 5%
    wind_direction_accuracy:                      3 degrees
    wind_speed_accuracy:                          0.3 m/s
    date:                                         2023-02-17